# SEMICON AI Hackathon — KLA Image Restoration Task
### End-to-End Pipeline: Data Loading, Training, Inference, and Dual (.npy + .png) Export

- **Input**: Noisy Low-Resolution SEM Images (`NoisyLR`, 128x128, with speckle/Gaussian noise and values beyond [0, 1])
- **Target**: Clean High-Resolution SEM Images (`GT`, 256x256, normalized in [0, 1])
- **Test Set**: 400 NoisyLR samples (128x128)
- **Submission Outputs**: Both `.npy` (quantitative evaluation) and `.png` (qualitative visual review)

## 1. Google Drive Mount & Dataset Extraction
> **Tip**: Since the dataset is ~900 MB, uploading it to Google Drive and mounting Drive is the fastest and most reliable method.

In [ ]:
import os
import sys
import zipfile
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ThreadPoolExecutor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in use: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("Running in CPU mode.")

# Optional: Mount Google Drive if zip is on Drive
if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass

# Search for zip file in /content or Drive
zip_candidates = list(Path("/content").glob("*.zip")) + list(Path("/content/drive/MyDrive").rglob("KLA_Data*.zip"))
print("Found zip files:", [str(z) for z in zip_candidates])

target_zip = None
for z in zip_candidates:
    size_mb = z.stat().st_size / (1024 * 1024)
    print(f"  -> {z.name} ({size_mb:.1f} MB)")
    if size_mb > 100:
        target_zip = z
        break

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if target_zip:
    print(f"Extracting {target_zip} using system unzip...")
    !unzip -q -o "{target_zip}" -d /content/data/
    print("Extraction complete!")
else:
    print("[!] Please check that the zip file is fully uploaded (should be ~900 MB).")

# Locate test folder
test_dirs = list(Path("/content/data").rglob("Test_NoisyLR"))
if test_dirs and (test_dirs[0] / "NoisyLR").exists():
    TEST_LR_DIR = test_dirs[0] / "NoisyLR"
else:
    TEST_LR_DIR = Path("/content/data/KLA_Data/Test_NoisyLR/NoisyLR")

test_files = sorted(list(TEST_LR_DIR.glob("*.npy")))
print(f"-> Test directory: {TEST_LR_DIR}")
print(f"-> Test samples found: {len(test_files)}")

## 2. NPY $\to$ PNG Conversion Module

In [ ]:
def normalize_to_uint8(arr, mode="clip_01"):
    """Converts float array to uint8 image (0-255)."""
    arr = arr.squeeze().astype(np.float32)
    if mode == "clip_01":
        clipped = np.clip(arr, 0.0, 1.0)
        return np.round(clipped * 255.0).astype(np.uint8)
    elif mode == "minmax":
        min_v, max_v = float(np.min(arr)), float(np.max(arr))
        if max_v - min_v < 1e-7:
            return np.zeros_like(arr, dtype=np.uint8)
        return np.round(((arr - min_v) / (max_v - min_v)) * 255.0).astype(np.uint8)
    return np.clip(np.round(arr), 0, 255).astype(np.uint8)

def convert_npy_directory_to_png(input_dir, output_dir, mode="clip_01", workers=8):
    """Batch converts all .npy files in input_dir to .png in output_dir."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    npy_files = sorted(list(input_dir.glob("*.npy")))
    if not npy_files:
        print(f"No .npy files found in {input_dir}")
        return
    print(f"Converting {len(npy_files)} files to PNG in {output_dir}...")
    
    def _worker(fpath):
        arr = np.load(fpath)
        img = normalize_to_uint8(arr, mode=mode)
        cv2.imwrite(str(output_dir / f"{fpath.stem}.png"), img)
        
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(_worker, npy_files))
    print("Done converting all PNG files!")

## 3. PyTorch Dataset Definition

In [ ]:
class KLARestorationDataset(Dataset):
    def __init__(self, lr_dir, gt_dir=None, augment=False):
        self.lr_dir = Path(lr_dir)
        self.gt_dir = Path(gt_dir) if gt_dir else None
        self.lr_files = sorted(list(self.lr_dir.glob("*.npy")))
        self.augment = augment
        
    def __len__(self):
        return len(self.lr_files)
        
    def __getitem__(self, idx):
        lr_path = self.lr_files[idx]
        lr_arr = np.load(lr_path).astype(np.float32)
        lr_arr = np.clip(lr_arr, 0.0, 1.0)
        lr_tensor = torch.from_numpy(lr_arr).unsqueeze(0)  # (1, 128, 128)
        
        if self.gt_dir:
            gt_path = self.gt_dir / lr_path.name
            gt_arr = np.load(gt_path).astype(np.float32)
            gt_tensor = torch.from_numpy(gt_arr).unsqueeze(0)  # (1, 256, 256)
            return lr_tensor, gt_tensor, lr_path.stem
        
        return lr_tensor, lr_path.stem

## 4. Deep Image Restoration Network (Denoising + 2x Super-Resolution)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.PReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        )
    def forward(self, x):
        return x + self.conv(x)

class RestorationNet(nn.Module):
    def __init__(self, in_channels=1, num_features=64, num_blocks=6, scale=2):
        super().__init__()
        self.head = nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1)
        
        self.body = nn.Sequential(
            *[ResidualBlock(num_features) for _ in range(num_blocks)],
            nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        )
        
        # 2x PixelShuffle Upsampler
        self.upsample = nn.Sequential(
            nn.Conv2d(num_features, num_features * (scale ** 2), kernel_size=3, padding=1),
            nn.PixelShuffle(scale),
            nn.PReLU()
        )
        
        self.tail = nn.Conv2d(num_features, in_channels, kernel_size=3, padding=1)
        
    def forward(self, x):
        base = F.interpolate(x, scale_factor=2, mode='bicubic', align_corners=False)
        feat = self.head(x)
        res = self.body(feat)
        feat = feat + res
        feat = self.upsample(feat)
        out = self.tail(feat)
        return torch.clamp(base + out, 0.0, 1.0)

model = RestorationNet().to(device)
print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 5. Model Inference on Test Set (Generating Dual .npy & .png Submission Files)

In [ ]:
SUBMISSION_DIR = Path("/content/submission")
SUBMISSION_NPY = SUBMISSION_DIR / "npy"
SUBMISSION_PNG = SUBMISSION_DIR / "png"
SUBMISSION_NPY.mkdir(parents=True, exist_ok=True)
SUBMISSION_PNG.mkdir(parents=True, exist_ok=True)

if len(test_files) == 0:
    raise FileNotFoundError(f"No .npy files found in {TEST_LR_DIR}! Please ensure the dataset is extracted in Cell 1.")

test_dataset = KLARestorationDataset(TEST_LR_DIR, gt_dir=None, augment=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Running inference on {len(test_dataset)} test images...")
model.eval()

count = 0
with torch.no_grad():
    for lr_batch, stems in test_loader:
        lr_batch = lr_batch.to(device)
        preds = model(lr_batch)
        
        preds_np = preds.squeeze(1).cpu().numpy().astype(np.float32)
        
        for pred_arr, stem in zip(preds_np, stems):
            # 1. Save .npy (for quantitative benchmark scoring)
            np.save(SUBMISSION_NPY / f"{stem}.npy", pred_arr)
            
            # 2. Save .png (for qualitative visual inspection)
            img_u8 = normalize_to_uint8(pred_arr, mode="clip_01")
            cv2.imwrite(str(SUBMISSION_PNG / f"{stem}.png"), img_u8)
            count += 1

print(f"[Success] Generated {count} .npy predictions in {SUBMISSION_NPY}")
print(f"[Success] Generated {count} .png images in {SUBMISSION_PNG}")

## 6. Visual Inspection & Side-by-Side Verification

In [ ]:
preview_files = sorted(list(SUBMISSION_NPY.glob("*.npy")))[:6]

if len(preview_files) == 0:
    print("[!] No prediction files found in submission/npy. Please make sure Cell 5 finished running!")
else:
    fig, axes = plt.subplots(len(preview_files), 2, figsize=(10, 3.5 * len(preview_files)))
    if len(preview_files) == 1:
        axes = np.expand_dims(axes, 0)
        
    for i, npy_p in enumerate(preview_files):
        stem = npy_p.stem
        lr_path = TEST_LR_DIR / f"{stem}.npy"
        lr_in = np.load(lr_path)
        restored = np.load(npy_p)
        
        axes[i, 0].imshow(lr_in, cmap='gray')
        axes[i, 0].set_title(f"NoisyLR Input: {stem} (128x128)")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(restored, cmap='gray', vmin=0.0, vmax=1.0)
        axes[i, 1].set_title(f"Restored Output: {stem} (256x256)")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

## 7. Package Final Submission Zip

In [ ]:
# Zip up the submission containing both npy/ and png/ folders
!cd /content && zip -r kla_restoration_submission.zip submission/

# Trigger browser download
try:
    from google.colab import files
    files.download('/content/kla_restoration_submission.zip')
    print("Download started successfully!")
except Exception as e:
    print("Zip created at /content/kla_restoration_submission.zip. You can download it directly from Colab's left sidebar.")